# Leonard — RFE, type matchups and logistic regression

We summarize team statistics and the first 30 battle turns, encode Pokémon/move identities, and use logistic-regression RFE to retain **300 features**. Adding **16 matchup** and **8 HP-based counter-quality features** gives **324 inputs** to a regularized logistic regression (`C=0.01`). Only official competition data and the supplied Gen-2 type chart are used.

Historical development CV accuracy: **67.40%** on five duplicate-grouped folds (7,996 rows). Repeated experiments used these folds, so the estimate may be optimistic. Run all cells to refit on all 10,000 training battles and generate 5,000 predictions. All feature and training logic is visible below; GitHub supplies the frozen configuration.


## 1. Frozen configuration
Use the local checkout or clone GitHub on Kaggle, then read the frozen settings and reference checksums. Publish the accompanying files to this revision before running on Kaggle.


In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import json

REPO_URL = "https://github.com/amrhsnd/FDS_Challenge_26.git"
REPO_REVISION = "leonard/feature-engineering-experiments"
USE_REFERENCE_RUNTIME = True

roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in roots if (p / "pyproject.toml").is_file()
                  and (p / "src/pokemon_challenge").is_dir()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path("/kaggle/working/FDS_Challenge_26")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REVISION], check=True)

FROZEN_MANIFEST = REPO_ROOT / "notebooks/leonard/frozen_submission.json"
if not FROZEN_MANIFEST.is_file():
    raise FileNotFoundError("Publish the frozen notebook and its accompanying source files to GitHub first.")
spec = json.loads(FROZEN_MANIFEST.read_text())
print("Repository:", REPO_ROOT)
print("Frozen model:", spec["model"])


## 2. Reference packages
Check package versions and install the reference versions if needed. Run from a fresh kernel; Kaggle needs Internet for cloning and installation.


In [ ]:
from importlib.metadata import version, PackageNotFoundError

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

mismatch = {name: expected for name, expected in spec["versions"].items()
            if installed_version(name) != expected}
if USE_REFERENCE_RUNTIME and mismatch:
    if any(name in sys.modules for name in ("numpy", "pandas", "scipy", "sklearn")):
        raise RuntimeError("Restart the kernel and run this notebook first to load reference dependencies.")
    runtime = Path("/kaggle/working/frozen-runtime") if Path("/kaggle/working").is_dir() else REPO_ROOT / "outputs/leonard/frozen-runtime"
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--target", str(runtime),
                    *[f"{name}=={expected}" for name, expected in spec["versions"].items()]], check=True)
    sys.path.insert(0, str(runtime))


## 3. Imports and file locations
Import the ML tools, locate the four official input files, and choose where to write the CSV. Set `POKEMON_DATA_DIR` to override input detection.


In [ ]:
from collections import Counter
import csv
import hashlib
import math
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import RFE, VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# Only the frozen JSON settings are read from the repository; all model code is below.
if os.environ.get('POKEMON_DATA_DIR'):
    DATA_DIR = Path(os.environ['POKEMON_DATA_DIR']).expanduser().resolve()
else:
    candidates = [
        Path('/kaggle/input/competitions/fds-pokemon-battles-prediction-2026'),
        Path('/kaggle/input/fds-pokemon-battles-prediction-2026'),
        REPO_ROOT / 'data/raw', REPO_ROOT / 'data',
    ]
    DATA_DIR = next((p for p in candidates if (p / 'train.jsonl').is_file()), None)
    if DATA_DIR is None:
        raise FileNotFoundError('Attach the competition data or set POKEMON_DATA_DIR.')
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else REPO_ROOT / 'outputs/leonard/frozen_submission'
print('Official inputs:', DATA_DIR)
print('Output folder:', OUTPUT_DIR)


## 4. Read and verify the data
Stream one battle at a time. Input checksums ensure this is the same official dataset used for the frozen submission.


In [ ]:
import json


from pathlib import Path


def iter_battles(path):
    """Yield one battle dictionary per nonempty JSONL line without changing it."""
    path = Path(path)
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                battle = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path}, line {line_number}") from error
            if not isinstance(battle, dict):
                raise ValueError(f"Expected a battle object in {path}, line {line_number}")
            yield battle


def sha256(path):
    """Hash files in chunks so large inputs need not be loaded into memory."""
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1048576), b''):
            checksum.update(block)
    return checksum.hexdigest()

for filename, expected in spec['input_sha256'].items():
    if sha256(DATA_DIR / filename) != expected:
        raise ValueError(f'Official input differs from the frozen run: {filename}')
print('All four official input checksums match.')


## 5. Starting team statistics
Aggregate our six Pokémon without relying on team order, scale base stats by level, and compare them with the opponent's known lead.


In [ ]:
import json


import pandas as pd


static_STATS = ["hp", "atk", "def", "spa", "spd", "spe"]


# The 17 real Gen-2 types. "notype" is a placeholder for "this Pokemon has
# only one type", NOT a real type, so it is excluded from the vocabulary.
static_NOT_A_TYPE = "notype"


def static_real_types(poke):
    """The Pokemon's actual types, with the 'notype' placeholder stripped out."""
    return [t for t in poke["types"] if t != static_NOT_A_TYPE]


def static_effective_stats(poke):
    """
    Base stats scaled by level.

    Base stats are the species' innate values and do NOT include level. 99.3%
    of Pokemon in this data are level 100, but 422 rows are not (abra, pineco,
    elekid, ...). Left unscaled, a level-5 abra looks exactly as strong as a
    level-100 abra, which is wrong.

    level/100 is a crude linear proxy for the real Gen-2 stat formula, which is
    roughly proportional to level. It is an approximation, but it is much closer
    to the truth than ignoring level entirely, and it costs nothing.
    """
    scale = poke["level"] / 100.0
    return {s: poke[f"base_{s}"] * scale for s in static_STATS}


def static_battle_to_row(battle, labelled):
    row = {"battle_id": battle["battle_id"]}

    team = battle["p1_team_details"]
    lead = battle["p2_lead_details"]

    # --- p1: six Pokemon collapsed into summary numbers -------------------
    # A team is a SET of six Pokemon with no meaningful order, so we must
    # aggregate. If we instead made columns like p1_poke1_atk ... p1_poke6_atk,
    # the model would learn position-specific patterns that are pure noise.
    team_stats = [static_effective_stats(p) for p in team]
    for s in static_STATS:
        vals = [ts[s] for ts in team_stats]
        row[f"p1_{s}_mean"] = sum(vals) / len(vals)   # overall team quality
        row[f"p1_{s}_max"] = max(vals)                # best specialist
    row["p1_level_mean"] = sum(p["level"] for p in team) / len(team)
    row["p1_level_min"] = min(p["level"] for p in team)

    # Type coverage: how many distinct real types the team spans. More coverage
    # means more answers to whatever p2 brings.
    row["p1_n_types"] = len({t for p in team for t in static_real_types(p)})
    # Dual-typed Pokemon get more resistances but also more weaknesses.
    row["p1_n_dual_type"] = sum(1 for p in team if len(static_real_types(p)) == 2)
    # 95.5% of items are 'leftovers'. A non-leftovers item is therefore a
    # deliberate, unusual choice and worth counting.
    row["p1_n_special_item"] = sum(1 for p in team if p["item"] != "leftovers")

    # --- p2: only the lead is known --------------------------------------
    # This is the asymmetry in the data. We cannot aggregate a team we have not
    # been shown, so for now we use the lead's raw numbers. Recovering the rest
    # of p2's team from the timeline is a later step.
    lead_stats = static_effective_stats(lead)
    for s in static_STATS:
        row[f"p2_lead_{s}"] = lead_stats[s]
    row["p2_lead_level"] = lead["level"]
    row["p2_lead_n_types"] = len(static_real_types(lead))

    # --- comparisons ------------------------------------------------------
    # Trees can in principle derive these, but handing a model the difference
    # directly makes the signal far easier to find than asking it to discover
    # "subtract column A from column B" by itself.
    for s in static_STATS:
        row[f"diff_{s}_mean"] = row[f"p1_{s}_mean"] - row[f"p2_lead_{s}"]
        row[f"diff_{s}_max"] = row[f"p1_{s}_max"] - row[f"p2_lead_{s}"]

    if labelled:
        row["player_won"] = battle["player_won"]
    return row


## 6. Timeline features
Summarize HP, faints, statuses, switches and observed moves. Active-Pokémon HP is distinct from whole-team HP.


In [ ]:
import json


import pandas as pd


timeline_ALIVE_STATUSES = {"nostatus"}   # 'fnt' means fainted; everything else is an ailment


timeline_EARLY = 10                       # turns 1-10


timeline_LATE = 20                        # turns 21-30


def timeline_side_features(turns, side):
    """
    Summarise one player's 30 turns. `side` is 'p1' or 'p2'.

    Note: each turn records only the ACTIVE Pokemon's state, not the whole team.
    So hp_pct jumps around as Pokemon switch in and out - it is the health of
    whoever is on the field, not a team health bar.
    """
    f = {}
    states = [t[f"{side}_pokemon_state"] for t in turns]
    moves = [t[f"{side}_move_details"] for t in turns]
    hp = [s["hp_pct"] for s in states]
    names = [s["name"] for s in states]

    # --- the scoreboard ---------------------------------------------------
    # Count DISTINCT species that fainted, not fainted turns. A species shows
    # as 'fnt' on exactly one turn then gets replaced (verified: 21,922 faint
    # events, all single-turn, and no species ever appears alive afterwards).
    f["faints"] = len({s["name"] for s in states if s["status"] == "fnt"})

    # --- health of the active Pokemon -------------------------------------
    f["hppct_mean"] = sum(hp) / len(hp)
    f["hppct_final"] = hp[-1]
    f["hppct_min"] = min(hp)
    # Momentum: is this side healthier or sicker than it was at the start?
    # A single snapshot cannot distinguish "steady at 60%" from "collapsing
    # from 100% to 60%", and those are very different positions.
    early = [h for h, t in zip(hp, turns) if t["turn"] <= timeline_EARLY]
    late = [h for h, t in zip(hp, turns) if t["turn"] > timeline_LATE]
    f["hppct_early_mean"] = sum(early) / len(early)
    f["hppct_late_mean"] = sum(late) / len(late)
    f["hppct_trend"] = f["hppct_late_mean"] - f["hppct_early_mean"]

    # --- status conditions ------------------------------------------------
    # Fraction of turns spent under an ailment (poison, sleep, paralysis...).
    # 'fnt' is excluded because being fainted is already counted above and is a
    # different kind of event from being debuffed while still fighting.
    ail = sum(1 for s in states if s["status"] not in timeline_ALIVE_STATUSES and s["status"] != "fnt")
    f["status_rate"] = ail / len(states)
    f["slp_rate"] = sum(1 for s in states if s["status"] == "slp") / len(states)

    # --- loss of control --------------------------------------------------
    # Switching: counts turns where the active species changed. Voluntary
    # switches are strategic, but forced ones (after a faint) mean you are
    # reacting rather than attacking.
    f["switches"] = sum(1 for a, b in zip(names, names[1:]) if a != b)
    f["n_species_seen"] = len(set(names))

    # Turns with NO move: move_details is null on ~20% of turn-sides. This is
    # structural, not missing data - it means no move happened (switched out,
    # fainted, asleep, recharging). It is deliberately NOT imputed; the rate
    # itself is the feature, because turns where you cannot act are turns the
    # opponent acts freely.
    f["no_move_rate"] = sum(1 for m in moves if m is None) / len(moves)

    # --- what they did when they could act --------------------------------
    acted = [m for m in moves if m is not None]
    if acted:
        f["mean_base_power"] = sum(m["base_power"] for m in acted) / len(acted)
        f["mean_accuracy"] = sum(m["accuracy"] for m in acted) / len(acted)
        # Status moves have base_power 0 - they buff, debuff or inflict
        # ailments instead of damaging. A high rate means a defensive game.
        f["status_move_rate"] = sum(1 for m in acted if m["category"] == "status") / len(acted)
    else:
        f["mean_base_power"] = 0.0
        f["mean_accuracy"] = 0.0
        f["status_move_rate"] = 0.0

    return f


def timeline_battle_to_row(battle):
    turns = battle["battle_timeline"]
    row = {"battle_id": battle["battle_id"]}
    for side in ("p1", "p2"):
        for k, v in timeline_side_features(turns, side).items():
            row[f"{side}_{k}"] = v

    # Differences, but ONLY where both sides carry signal. Deliberately NOT
    # including a faint difference: p2's faint count is flat across outcomes
    # (0.49 when p1 loses vs 0.52 when p1 wins), so subtracting it would add
    # noise to p1's faint count, which does carry signal (1.98 vs 1.40).
    for k in ("hppct_mean", "hppct_final", "status_rate", "no_move_rate", "switches"):
        row[f"diff_{k}"] = row[f"p1_{k}"] - row[f"p2_{k}"]
    return row


## 7. Effects and boosts
Measure how often ongoing effects occur and summarize temporary offensive, defensive and speed boosts.


In [ ]:
import pandas as pd


# The effects common enough to be worth a column of their own. Rarer ones
# (bide, mindreader, spite...) appear a handful of times in 300,000 turns
# and would be mostly zeros.
effects_TRACKED_EFFECTS = [
    "substitute",
    "trapped",
    "confusion",
    "leechseed",
    "protect",
    "perish",
    "attract",
    "encore",
    "destinybond",
    "clamp",
]


# The five stats that can be boosted during a battle.
effects_BOOST_STATS = ["atk", "def", "spa", "spd", "spe"]


def effects_side_features(turns, side):
    """Summarise one player's effects and boosts across the 30 turns."""
    features = {}
    states = [turn[f"{side}_pokemon_state"] for turn in turns]
    n_turns = len(states)

    # --- effects -------------------------------------------------------
    # For each tracked effect, what fraction of the 30 turns was it active?
    for effect in effects_TRACKED_EFFECTS:
        count = 0
        for state in states:
            if effect in state["effects"]:
                count += 1
        features[f"eff_{effect}"] = count / n_turns

    # How often ANY effect was active, as a general measure of how eventful
    # this player's side of the battle was.
    any_effect = 0
    for state in states:
        real_effects = [e for e in state["effects"] if e != "noeffect"]
        if real_effects:
            any_effect += 1
    features["eff_any"] = any_effect / n_turns

    # --- boosts --------------------------------------------------------
    # Each turn, add up the five stat stages. Positive means this Pokemon
    # has been buffed overall; negative means it has been weakened.
    totals = []
    offensive = []
    defensive = []
    speed = []

    for state in states:
        boosts = state["boosts"]
        totals.append(sum(boosts[s] for s in effects_BOOST_STATS))
        offensive.append(boosts["atk"] + boosts["spa"])
        defensive.append(boosts["def"] + boosts["spd"])
        speed.append(boosts["spe"])

    features["boost_mean"] = sum(totals) / n_turns
    features["boost_max"] = max(totals)
    features["boost_final"] = totals[-1]
    features["boost_offensive_mean"] = sum(offensive) / n_turns
    features["boost_defensive_mean"] = sum(defensive) / n_turns
    features["boost_speed_mean"] = sum(speed) / n_turns

    return features


def effects_battle_to_row(battle):
    row = {"battle_id": battle["battle_id"]}
    turns = battle["battle_timeline"]

    for side in ("p1", "p2"):
        for name, value in effects_side_features(turns, side).items():
            row[f"{side}_{name}"] = value

    # Differences for the handful with the clearest signal. We keep this
    # short on purpose: adding a difference for every column has repeatedly
    # hurt us, because subtracting a noisy column from a useful one just
    # spreads the noise.
    for name in ("eff_substitute", "eff_trapped", "boost_mean"):
        row[f"diff_{name}"] = row[f"p1_{name}"] - row[f"p2_{name}"]

    return row


## 8. Remaining team health
Track last-seen HP and team snapshots at turns 10, 20 and 30. The baseline provisionally treats unseen team slots as healthy.


In [ ]:
import pandas as pd


roster_TEAM_SIZE = 6


# Points in the battle where we take a snapshot of the scoreboard, so the
# model can see the trend and not just the end state.
roster_CHECKPOINTS = [10, 20, 30]


def roster_track_rosters(battle):
    """
    Replay the 30 turns and record each side's team state at the checkpoints.

    Returns two dictionaries, one per side, of the form
        {turn: (number_alive, total_hp)}
    """
    # Player 1's whole team is known, so start everyone at full health.
    p1_roster = {p["name"]: 1.0 for p in battle["p1_team_details"]}

    # For player 2 we only know the lead. Others get added as they appear.
    p2_roster = {battle["p2_lead_details"]["name"]: 1.0}

    p1_snapshots = {}
    p2_snapshots = {}

    for turn in battle["battle_timeline"]:
        # Update whoever is currently on the field with their latest health.
        s1 = turn["p1_pokemon_state"]
        s2 = turn["p2_pokemon_state"]
        p1_roster[s1["name"]] = s1["hp_pct"]
        p2_roster[s2["name"]] = s2["hp_pct"]

        if turn["turn"] in roster_CHECKPOINTS:
            p1_snapshots[turn["turn"]] = roster_summarise(p1_roster)
            p2_snapshots[turn["turn"]] = roster_summarise(p2_roster)

    return p1_snapshots, p2_snapshots


def roster_summarise(roster):
    """
    Turn a roster into (how many are alive, total health of the team).

    Pokemon we have never seen count as alive and at full health, which is
    what "not yet sent out" means.
    """
    never_seen = max(0, roster_TEAM_SIZE - len(roster))

    alive = never_seen
    total_hp = float(never_seen)
    for hp in roster.values():
        if hp > 0:
            alive += 1
        total_hp += hp

    return alive, total_hp


def roster_battle_to_row(battle):
    row = {"battle_id": battle["battle_id"]}
    p1_snapshots, p2_snapshots = roster_track_rosters(battle)

    for turn in roster_CHECKPOINTS:
        p1_alive, p1_hp = p1_snapshots[turn]
        p2_alive, p2_hp = p2_snapshots[turn]

        row[f"p1_alive_t{turn}"] = p1_alive
        row[f"p2_alive_t{turn}"] = p2_alive
        row[f"alive_diff_t{turn}"] = p1_alive - p2_alive

        row[f"p1_teamhp_t{turn}"] = p1_hp
        row[f"p2_teamhp_t{turn}"] = p2_hp
        row[f"teamhp_diff_t{turn}"] = p1_hp - p2_hp

    # How fast each side is losing ground. A team that went from 6.0 to 3.0
    # total HP is collapsing; one that went 6.0 to 5.5 is holding.
    row["p1_teamhp_decline"] = row["p1_teamhp_t10"] - row["p1_teamhp_t30"]
    row["p2_teamhp_decline"] = row["p2_teamhp_t10"] - row["p2_teamhp_t30"]
    row["teamhp_decline_diff"] = row["p1_teamhp_decline"] - row["p2_teamhp_decline"]

    return row


## 9. Recent battle state
Track roster statuses, observation age and recent HP changes. Compare consecutive HP readings only when the active species stays the same.


In [ ]:
def battle_state_features(battle):
    """Track last-seen roster status and recent changes without switch HP jumps.

    Unobserved opponents are not assigned a status. Last-seen HP/status can be
    stale; exposure and observation-age features describe that uncertainty.
    """
    turns = [t for t in battle['battle_timeline'] if t['turn'] <= 30]
    if not turns:
        raise ValueError('At least one observed turn is required')
    row = {}
    for side in ('p1', 'p2'):
        seen = {}
        for t in turns:
            state = t[f'{side}_pokemon_state']
            seen[state['name']] = (state, t['turn'])
        states = [s for s, _ in seen.values()]
        alive = [s for s in states if s['hp_pct'] > 0 and s['status'] != 'fnt']
        values = {
            'observed_alive': len(alive),
            'observed_injured': sum(0 < s['hp_pct'] < .5 for s in states),
            'observed_hp_sq': sum(s['hp_pct'] ** 2 for s in states),
            'last_seen_age_mean': sum(turns[-1]['turn'] - t for _, t in seen.values()) / len(seen),
        }
        for status in ('slp', 'par', 'brn', 'psn', 'tox', 'frz'):
            values[f'roster_{status}'] = sum(s['status'] == status for s in alive)
            values[f'final_{status}'] = int(turns[-1][f'{side}_pokemon_state']['status'] == status)
        for window in (5, 10):
            recent = turns[-window:]
            recent_states = [t[f'{side}_pokemon_state'] for t in recent]
            values[f'recent_{window}_faints'] = len({s['name'] for s in recent_states if s['status'] == 'fnt'})
            values[f'recent_{window}_no_move'] = sum(t[f'{side}_move_details'] is None for t in recent) / len(recent)
            # Compare consecutive observations only for the same active species.
            changes = [b['hp_pct'] - a['hp_pct'] for a, b in zip(recent_states, recent_states[1:]) if a['name'] == b['name']]
            values[f'recent_{window}_hp_loss'] = sum(max(-v, 0) for v in changes)
            values[f'recent_{window}_hp_gain'] = sum(max(v, 0) for v in changes)
        for stat in ('atk', 'def', 'spa', 'spd', 'spe'):
            values[f'final_boost_{stat}'] = turns[-1][f'{side}_pokemon_state']['boosts'][stat]
        row.update({f'{side}_{key}': value for key, value in values.items()})
    for key in values:
        row[f'diff_{key}'] = row[f'p1_{key}'] - row[f'p2_{key}']
    return row


## 10. Boost threats
Combine attack, special-attack and speed boosts with HP, status and observation recency. These are threat proxies, not damage calculations.


In [ ]:
STATS = ('hp', 'atk', 'def', 'spa', 'spd', 'spe')


def observed_turns(battle):
    turns = sorted((t for t in battle['battle_timeline'] if 1 <= t['turn'] <= 30), key=lambda t: t['turn'])
    if not turns:
        raise ValueError('No observed turns in 1–30')
    return turns


def last_seen(turns, side):
    seen = {}
    for t in turns:
        seen[t[f'{side}_pokemon_state']['name']] = (t[f'{side}_pokemon_state'], t['turn'])
    return seen


def boost_threat(battle):
    """Separate attack types, HP-weighted stages and status interactions.

    Last-seen boosts may reset on switches; age and active flags distinguish
    stale bench observations. Values are observations, not exact damage estimates.
    """
    turns = observed_turns(battle)
    row = {}
    for side in ('p1', 'p2'):
        seen = last_seen(turns, side)
        active = turns[-1][f'{side}_pokemon_state']
        alive = [(s, t) for s, t in seen.values() if s['hp_pct'] > 0 and s['status'] != 'fnt']
        for stat in ('atk', 'spa', 'spe'):
            stage = active['boosts'][stat]
            hp = 0. if active['status'] == 'fnt' else active['hp_pct']
            row[f'{side}_active_{stat}_stage'] = stage
            row[f'{side}_active_{stat}_hp_stage'] = hp * stage
            for status in ('slp', 'par', 'brn'):
                row[f'{side}_active_{stat}_{status}_interaction'] = hp * stage * (active['status'] == status)
            threats = [max(s['boosts'][stat], 0) * s['hp_pct'] for s, _ in alive]
            row[f'{side}_observed_{stat}_threat_max'] = max(threats, default=0.)
            row[f'{side}_observed_{stat}_threat_sum'] = sum(threats)
            row[f'{side}_recent_{stat}_threat_max'] = max((max(s['boosts'][stat], 0)*s['hp_pct'] for s, t in alive if turns[-1]['turn']-t <= 5), default=0.)
    for key in list(row):
        if key.startswith('p1_'):
            row['diff_' + key[3:]] = row[key] - row['p2_' + key[3:]]
    return row


## 11. Assemble numeric inputs
Use only supplied metadata and turns 1–30, exclude IDs/labels, and keep the original feature names and ordering.


In [ ]:
def feature_row(battle):
    """Combine the numeric feature families used by the frozen baseline."""
    observed = {k: battle[k] for k in ('p1_team_details', 'p2_lead_details', 'battle_timeline')}
    observed['battle_timeline'] = [t for t in observed['battle_timeline'] if 1 <= t['turn'] <= 30]
    observed['battle_id'] = 0  # Compatibility with row builders; never a model feature.

    def clean(values):
        return {k: v for k, v in values.items() if k not in ('battle_id', 'player_won')}

    effect_values = clean(effects_battle_to_row(observed))
    original = {
        **clean(static_battle_to_row(observed, labelled=False)),
        **clean(timeline_battle_to_row(observed)),
        **{k: v for k, v in effect_values.items() if 'boost' in k},
        **clean(roster_battle_to_row(observed)),
    }
    assert len(original) == 106
    # Other experimental feature families are omitted: the frozen model never uses them.
    groups = {
        'original': original,
        'effects': {k: v for k, v in effect_values.items() if 'eff_' in k},
        'battle_state': battle_state_features(observed),
        'boost_threat': boost_threat(observed),
    }
    return {f'{group}__{key}': value for group, values in groups.items() for key, value in values.items()}


## 12. Pokémon and move identities
Encode observed species, moves, usage and recency. The encoder learns its vocabulary during fitting and ignores unknown identities at prediction time.


In [ ]:
from collections import Counter


import numpy as np


import pandas as pd


from sklearn.base import BaseEstimator, TransformerMixin


from sklearn.feature_extraction import DictVectorizer


CAT_KEYS=('p1_active','p2_active','p2_lead','p1_last_move','p2_last_move',
          'p1_most_used_move','p2_most_used_move')


def identity_features(battle):
    """No IDs/targets. Unseen opponents stay unknown; own unseen HP defaults to 1.

    Per-species last HP can be stale. Move names are observations, not assumed
    complete movesets. Dynamic dictionaries are encoded using training-fold-only
    vocabularies; new species/moves at inference are safely ignored by that encoder.
    """
    turns=sorted((t for t in battle['battle_timeline'] if 1 <= t['turn'] <= 30),key=lambda t:t['turn'])
    if not turns: raise ValueError('No observed turns')
    numeric={};categorical={'p2_lead':battle['p2_lead_details']['name']}
    for side in ('p1','p2'):
        seen={};species_count=Counter();moves=Counter();recent=Counter();last_used={};types=Counter()
        for t in turns:
            state=t[f'{side}_pokemon_state'];name=state['name']
            seen[name]=state;species_count[name]+=1
            move=t[f'{side}_move_details']
            if move is not None:
                moves[move['name']]+=1;last_used[move['name']]=t['turn'];types[move['type']]+=1
                if t['turn'] > turns[-1]['turn']-5:recent[move['name']]+=1
        for name,state in seen.items():
            numeric[f'{side}_seen_species::{name}']=1.
            numeric[f'{side}_species_exposure::{name}']=species_count[name]/len(turns)
            numeric[f'{side}_last_hp::{name}']=0. if state['status']=='fnt' else state['hp_pct']
        for name,count in moves.items():
            numeric[f'{side}_move_rate::{name}']=count/len(turns)
            numeric[f'{side}_recent_move_rate::{name}']=recent[name]/min(5,len(turns))
            numeric[f'{side}_move_recency::{name}']=1./(1+turns[-1]['turn']-last_used[name])
        for name,count in types.items():numeric[f'{side}_move_type_rate::{name}']=count/len(turns)
        categorical[f'{side}_active']=turns[-1][f'{side}_pokemon_state']['name']
        categorical[f'{side}_last_move']=max(last_used,key=last_used.get) if last_used else '<none>'
        categorical[f'{side}_most_used_move']=min(moves,key=lambda n:(-moves[n],n)) if moves else '<none>'
        if side=='p1':
            for poke in battle['p1_team_details']:
                name=poke['name'];state=seen.get(name)
                numeric[f'p1_team_species::{name}']=1.
                numeric[f'p1_team_species_hp::{name}']=1. if state is None else (0. if state['status']=='fnt' else state['hp_pct'])
    assert set(categorical)==set(CAT_KEYS)
    return {'identity_numeric':numeric,**categorical}


class IdentityEncoder(TransformerMixin,BaseEstimator):
    """Fold-fitted one-hot species/move vocabulary plus fixed numeric features."""
    def __init__(self,base_columns):
        self.base_columns=base_columns

    def _dictionaries(self,X):
        rows=[]
        for row in X[['identity_numeric',*CAT_KEYS]].to_dict('records'):
            values=dict(row['identity_numeric'])
            values.update({k:row[k] for k in CAT_KEYS})
            rows.append(values)
        return rows

    # Learn the identity vocabulary from the fitting data only.
    def fit(self,X,y=None):
        self.vectorizer_=DictVectorizer(sparse=False,dtype=np.float64)
        self.vectorizer_.fit(self._dictionaries(X))
        self.numeric_columns_=[f'n{i}' for i in range(len(self.base_columns)+len(self.vectorizer_.feature_names_))]
        return self

    def transform(self,X):
        identity=self.vectorizer_.transform(self._dictionaries(X))
        numbers=np.column_stack([X[self.base_columns].to_numpy(dtype=float),identity])
        if not np.isfinite(numbers).all():raise ValueError('Nonfinite model input')
        return pd.DataFrame(numbers,columns=self.numeric_columns_,index=X.index)

    def get_feature_names_out(self,input_features=None):
        return np.asarray(self.numeric_columns_,dtype=object)


## 13. Type chart and observed information
Read the supplied sparse Gen-2 chart; missing valid pairs are neutral and dual-type multipliers multiply. Record only revealed opponents and usable observed move types.


In [ ]:
import csv


import math


import numpy as np


import pandas as pd


from sklearn.base import BaseEstimator,TransformerMixin


TYPES=frozenset(('normal','fire','water','electric','grass','ice','fighting','poison',
                 'ground','flying','psychic','bug','rock','ghost','dragon','dark','steel'))


ACTIVE_KEYS=('p1_active_stab_best','p2_active_stab_best','active_stab_advantage',
             'p1_active_observed_move_best','p2_active_observed_move_best','active_observed_move_advantage',
             'p1_active_types_known','p2_active_types_known','p1_active_moves_known','p2_active_moves_known')


TEAM_KEYS=('favorable_pair_fraction','resistant_pair_fraction','immune_pair_fraction',
           'hp_weighted_offense','hp_weighted_enemy_offense','hp_weighted_advantage',
           'counter_coverage_fraction','safe_counter_coverage_fraction',
           'observed_move_favorable_pair_fraction','observed_move_resistance_pair_fraction',
           'observed_move_immunity_pair_fraction','hp_weighted_observed_move_advantage',
           'p2_revealed_type_coverage','p1_known_move_coverage','p2_known_move_coverage','p2_unseen_count')


def read_chart(path):
    chart={}
    with open(path) as stream:
        for row in csv.DictReader(stream):
            key=(row['attacking_type'],row['defending_type'])
            if key in chart:raise ValueError('Duplicate chart entry')
            value=float(row['multiplier'])
            if not set(key)<=TYPES or value not in (0.,.5,1.,2.):raise ValueError('Invalid Gen-2 chart entry')
            chart[key]=value
    return chart


def real_types(poke):
    types=tuple(sorted(t for t in poke.get('types',()) if t!='notype'))
    return types if types and set(types)<=TYPES else None


def effectiveness(attack_type,defending_types,chart):
    if attack_type not in TYPES or not defending_types or not set(defending_types)<=TYPES:return np.nan
    # The supplied chart lists non-neutral entries only; missing valid pairs = 1.
    return math.prod(chart.get((attack_type,t),1.) for t in defending_types)


def best_effect(attack_types,defending_types,chart):
    values=[effectiveness(t,defending_types,chart) for t in attack_types]
    values=[v for v in values if np.isfinite(v)]
    return max(values) if values else np.nan


def battle_metadata(battle):
    return {p['name']:real_types(p) for p in [*battle['p1_team_details'],battle['p2_lead_details']]}


def _mean(values):
    return float(np.mean(values)) if values else np.nan


def _weighted(values):
    # [(value, weight)]; NaNs express unavailable knowledge, not neutral matchups.
    valid=[(v,w) for v,w in values if np.isfinite(v) and w>0]
    return sum(v*w for v,w in valid)/sum(w for _,w in valid) if valid else np.nan


def summarize_battle(battle):
    turns=sorted((t for t in battle['battle_timeline'] if 1<=t['turn']<=30),key=lambda t:t['turn'])
    if not turns:raise ValueError('No observed turns')
    seen={'p1':{},'p2':{}};moves={'p1':{},'p2':{}}
    for t in turns:
        for side in ('p1','p2'):
            s=t[f'{side}_pokemon_state'];seen[side][s['name']]=s
            move=t[f'{side}_move_details']
            # Unknown actual Hidden Power type and zero-power/fixed-damage moves
            # are omitted. These are type-coverage proxies, not damage estimates.
            if (move and move['category'] in ('physical','special') and move['base_power']>0
                    and move['name']!='hiddenpower' and move['type'] in TYPES):
                moves[side].setdefault(s['name'],set()).add(move['type'])
    active={side:turns[-1][f'{side}_pokemon_state']['name'] for side in ('p1','p2')}
    return {'_matchup_summary':True,
            'p1_team_details':[{'name':p['name'],'types':p['types']} for p in battle['p1_team_details']],
            'p2_lead_details':{'name':battle['p2_lead_details']['name'],'types':battle['p2_lead_details']['types']},
            'seen':{side:{name:{k:s[k] for k in ('name','hp_pct','status')} for name,s in states.items()} for side,states in seen.items()},
            'moves':{side:{name:sorted(values) for name,values in by_species.items()} for side,by_species in moves.items()},
            'active':active}


## 14. Team-matchup features
Calculate favorable pairings, resistances, HP-weighted advantages and counter coverage. The final model uses the 16 `TEAM_KEYS`; active-only summaries are not selected.


In [ ]:
def extract_matchups(battle,species_types,chart):
    battle=battle if battle.get('_matchup_summary') is True else summarize_battle(battle)
    types={**species_types,**battle_metadata(battle)}
    seen=battle['seen'];moves=battle['moves'];active=battle['active']
    row={}
    for side,other in [('p1','p2'),('p2','p1')]:
        own=active[side];enemy=active[other]
        row[f'{side}_active_stab_best']=best_effect(types.get(own) or (),types.get(enemy),chart)
        row[f'{side}_active_observed_move_best']=best_effect(moves[side].get(own,set()),types.get(enemy),chart)
        row[f'{side}_active_types_known']=float(bool(types.get(own)))
        row[f'{side}_active_moves_known']=float(bool(moves[side].get(own)))
    row['active_stab_advantage']=row['p1_active_stab_best']-row['p2_active_stab_best']
    row['active_observed_move_advantage']=row['p1_active_observed_move_best']-row['p2_active_observed_move_best']
    own=[]
    for p in battle['p1_team_details']:
        s=seen['p1'].get(p['name']);hp=1. if s is None else (0. if s['status']=='fnt' else s['hp_pct'])
        if hp>0:own.append((p['name'],hp))
    enemies=[(name,s['hp_pct']) for name,s in seen['p2'].items() if s['hp_pct']>0 and s['status']!='fnt']
    favorable=[];resistant=[];immune=[];offense=[];enemy_offense=[];advantage=[]
    move_favorable=[];move_resistant=[];move_immune=[];move_advantage=[]
    coverage=[];safe=[]
    # Evaluate each living own Pokémon against each revealed living enemy.
    for enemy,ehp in enemies:
        favorable_counters=[];safe_counters=[]
        for name,hp in own:
            a=best_effect(types.get(name) or (),types.get(enemy),chart)
            b=best_effect(types.get(enemy) or (),types.get(name),chart)
            weight=hp*ehp
            if np.isfinite(a) and np.isfinite(b):
                favorable.append(float(a>1));resistant.append(float(b<1));immune.append(float(b==0))
                offense.append((a,weight));enemy_offense.append((b,weight));advantage.append((a-b,weight))
                favorable_counters.append(a>1);safe_counters.append(a>1 and b<1)
            ma=best_effect(moves['p1'].get(name,set()),types.get(enemy),chart)
            mb=best_effect(moves['p2'].get(enemy,set()),types.get(name),chart)
            if np.isfinite(ma):move_favorable.append(float(ma>1))
            if np.isfinite(mb):move_resistant.append(float(mb<1));move_immune.append(float(mb==0))
            if np.isfinite(ma) and np.isfinite(mb):move_advantage.append((ma-mb,weight))
        if favorable_counters:coverage.append(float(any(favorable_counters)));safe.append(float(any(safe_counters)))
    row.update(favorable_pair_fraction=_mean(favorable),resistant_pair_fraction=_mean(resistant),
               immune_pair_fraction=_mean(immune),hp_weighted_offense=_weighted(offense),
               hp_weighted_enemy_offense=_weighted(enemy_offense),hp_weighted_advantage=_weighted(advantage),
               counter_coverage_fraction=_mean(coverage),safe_counter_coverage_fraction=_mean(safe),
               observed_move_favorable_pair_fraction=_mean(move_favorable),
               observed_move_resistance_pair_fraction=_mean(move_resistant),
               observed_move_immunity_pair_fraction=_mean(move_immune),
               hp_weighted_observed_move_advantage=_weighted(move_advantage),
               p2_revealed_type_coverage=sum(bool(types.get(n)) for n,_ in enemies)/max(1,len(enemies)),
               p1_known_move_coverage=sum(bool(moves['p1'].get(n)) for n,_ in own)/max(1,len(own)),
               p2_known_move_coverage=sum(bool(moves['p2'].get(n)) for n,_ in enemies)/max(1,len(enemies)),
               p2_unseen_count=float(max(0,6-len(seen['p2']))))
    assert set(row)==set(ACTIVE_KEYS+TEAM_KEYS)
    return row


class MatchupTransformer(TransformerMixin,BaseEstimator):
    def __init__(self,chart_path,group='all'):
        self.chart_path=chart_path;self.group=group

    # Learn species types from fitting metadata; never consult winner labels.
    def fit(self,X,y=None):
        if self.group not in ('active','team','all'):raise ValueError('Unknown matchup group')
        self.chart_=read_chart(self.chart_path);self.species_types_={}
        for battle in X['battle']:
            for name,types in battle_metadata(battle).items():
                if types is None:continue
                if name in self.species_types_ and self.species_types_[name]!=types:raise ValueError('Conflicting species types')
                self.species_types_[name]=types
        self.feature_names_=list(ACTIVE_KEYS if self.group=='active' else TEAM_KEYS if self.group=='team' else ACTIVE_KEYS+TEAM_KEYS)
        return self

    def transform(self,X):
        rows=[extract_matchups(b,self.species_types_,self.chart_) for b in X['battle']]
        return pd.DataFrame(rows,index=X.index)[self.feature_names_]

    def get_feature_names_out(self,input_features=None):
        return np.asarray(self.feature_names_,dtype=object)


## 15. HP-based counter quality
A counter has type effectiveness >1; a safe counter also resists the enemy's type options (<1). The model uses the 8 `QUALITY_KEYS`, including best counter HP, HP reserves and uncovered threats.


In [ ]:
import numpy as np


import pandas as pd


from sklearn.base import BaseEstimator,TransformerMixin


REDUNDANCY_KEYS=('counter_count_mean','counter_count_min','safe_counter_count_mean',
                'safe_counter_count_min','two_counter_coverage_fraction','two_safe_counter_coverage_fraction',
                'no_safe_counter_enemy_count','single_safe_counter_enemy_fraction')


QUALITY_KEYS=('best_counter_hp_mean','weakest_best_counter_hp','best_safe_counter_hp_mean',
              'weakest_best_safe_counter_hp','healthy_safe_counter_coverage_fraction',
              'counter_hp_reserve_mean','safe_counter_hp_reserve_mean','hp_weighted_uncovered_threat_fraction')


def extract_counters(battle,species_types,chart):
    battle=battle if battle.get('_matchup_summary') is True else summarize_battle(battle)
    types={**species_types,**battle_metadata(battle)};seen=battle['seen']
    own=[]
    for p in battle['p1_team_details']:
        s=seen['p1'].get(p['name']);hp=1. if s is None else (0. if s['status']=='fnt' else s['hp_pct'])
        if hp>0 and types.get(p['name']):own.append((p['name'],hp))
    own_alive=sum(1 for p in battle['p1_team_details'] if
                  p['name'] not in seen['p1'] or
                  (seen['p1'][p['name']]['hp_pct']>0 and seen['p1'][p['name']]['status']!='fnt'))
    enemies=[(name,s['hp_pct']) for name,s in seen['p2'].items()
             if s['hp_pct']>0 and s['status']!='fnt' and types.get(name)]
    # No known matchup does not mean no counter. Return missing values when all
    # revealed enemies have unknown types (or all alive own types are unknown).
    if not enemies or (own_alive and not own):
        return dict.fromkeys(REDUNDANCY_KEYS+QUALITY_KEYS,np.nan)
    counters=[];safe_counts=[];best_hp=[];best_safe_hp=[];reserves=[];safe_reserves=[];uncovered=[]
    # For each enemy, collect HP reserves in favorable and safe own counters.
    for enemy,enemy_hp in enemies:
        good=[];safe=[]
        for name,hp in own:
            a=best_effect(types.get(name) or (),types.get(enemy),chart)
            b=best_effect(types.get(enemy) or (),types.get(name),chart)
            if np.isfinite(a) and a>1:
                good.append(hp)
                if np.isfinite(b) and b<1:safe.append(hp)
        counters.append(len(good));safe_counts.append(len(safe))
        best_hp.append(max(good,default=0.));best_safe_hp.append(max(safe,default=0.))
        reserves.append(sum(good));safe_reserves.append(sum(safe))
        uncovered.append((float(not safe),enemy_hp))
    # Keep both helper groups for parity with the tested extractor.
    # Only QUALITY_KEYS enter this frozen model.
    row={'counter_count_mean':_mean(counters),'counter_count_min':float(min(counters)),
         'safe_counter_count_mean':_mean(safe_counts),'safe_counter_count_min':float(min(safe_counts)),
         'two_counter_coverage_fraction':_mean([float(n>=2) for n in counters]),
         'two_safe_counter_coverage_fraction':_mean([float(n>=2) for n in safe_counts]),
         'no_safe_counter_enemy_count':float(sum(n==0 for n in safe_counts)),
         'single_safe_counter_enemy_fraction':_mean([float(n==1) for n in safe_counts]),
         'best_counter_hp_mean':_mean(best_hp),'weakest_best_counter_hp':min(best_hp),
         'best_safe_counter_hp_mean':_mean(best_safe_hp),'weakest_best_safe_counter_hp':min(best_safe_hp),
         'healthy_safe_counter_coverage_fraction':_mean([float(hp>=.5) for hp in best_safe_hp]),
         'counter_hp_reserve_mean':_mean(reserves),'safe_counter_hp_reserve_mean':_mean(safe_reserves),
         'hp_weighted_uncovered_threat_fraction':_weighted(uncovered)}
    assert set(row)==set(REDUNDANCY_KEYS+QUALITY_KEYS)
    return row


class CounterTransformer(TransformerMixin,BaseEstimator):
    def __init__(self,chart_path,group='all'):
        self.chart_path=chart_path;self.group=group

    def fit(self,X,y=None):
        if self.group not in ('redundancy','quality','all'):raise ValueError('Unknown counter group')
        self.lookup_=MatchupTransformer(self.chart_path,'team').fit(X,y)
        self.feature_names_=list(REDUNDANCY_KEYS if self.group=='redundancy' else
                                 QUALITY_KEYS if self.group=='quality' else REDUNDANCY_KEYS+QUALITY_KEYS)
        return self

    def transform(self,X):
        rows=[extract_counters(b,self.lookup_.species_types_,self.lookup_.chart_) for b in X['battle']]
        return pd.DataFrame(rows,index=X.index)[self.feature_names_]

    def get_feature_names_out(self,input_features=None):
        return np.asarray(self.feature_names_,dtype=object)


## 16. Frozen model pipeline
RFE keeps 300 baseline/identity inputs. Separately impute and scale the 16 matchup and 8 quality inputs, concatenate all branches, and apply strongly regularized logistic regression.


In [ ]:
def classifier():
    return LogisticRegression(C=.01, max_iter=4000, solver='lbfgs', random_state=42)


def make_frozen_model(columns, chart_path):
    # Standardize only the 289 numeric inputs; leave identity indicators/rates raw.
    # Remove constants, then eliminate weak coefficients until 300 inputs remain.
    baseline = Pipeline([
        ('encoding', IdentityEncoder(columns)),
        ('scaling', ColumnTransformer([
            ('base', StandardScaler(), list(range(len(columns))))
        ], remainder='passthrough')),
        ('constant_filter', VarianceThreshold()),
        ('feature_selection', RFE(classifier(), n_features_to_select=300, step=.25)),
    ])
    # Matchup and counter branches each fit their own imputation and scaling.
    def additions(transformer):
        return Pipeline([
            ('raw', transformer),
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('constant_filter', VarianceThreshold()),
            ('scaler', StandardScaler()),
        ])
    # Keep the RFE300 baseline intact and append the selected 16 + 8 additions.
    current = FeatureUnion([
        ('baseline', baseline),
        ('matchups', additions(MatchupTransformer(str(chart_path), 'team'))),
    ])
    return Pipeline([
        ('features', FeatureUnion([
            ('current', current),
            ('counters', additions(CounterTransformer(str(chart_path), 'quality'))),
        ])),
        ('model', classifier()),
    ])


## 17. Build train and test matrices
Extract all selected numeric inputs, identity dictionaries and observed matchup summaries. Battle IDs are retained only for aligning the final CSV.


In [ ]:
def build_inputs(path, columns, labelled=False):
    rows, ids, labels = [], [], []
    for i, battle in enumerate(iter_battles(path), 1):
        numeric = feature_row(battle)
        rows.append({**{k: numeric[k] for k in columns},
                     **identity_features(battle), 'battle': summarize_battle(battle)})
        ids.append(battle['battle_id'])
        if labelled:
            labels.append(int(battle['player_won']))
        if i % 2000 == 0:
            print(f'{Path(path).name}: extracted {i} battles', flush=True)
    if len(ids) != len(set(ids)) or not ids:
        raise ValueError('Empty data or duplicate battle IDs')
    return pd.DataFrame(rows), ids, np.asarray(labels, dtype=int)


columns = spec['numeric_columns']
X, train_ids, y = build_inputs(DATA_DIR / 'train.jsonl', columns, labelled=True)
test, test_ids, _ = build_inputs(DATA_DIR / 'test.jsonl', columns)
assert len(columns) == 289
assert len(y) == 10000 and len(test_ids) == 5000 and set(y) == {0, 1}


## 18. Train and predict
Fit the frozen procedure on all training labels and predict the test battles. No model search or cross-validation is performed in this submission run.


In [ ]:
# Limit numerical-library threads for a reproducible, modest CPU workload.
with threadpool_limits(limits=2):
    model = make_frozen_model(columns, DATA_DIR / 'type_chart_gen2.csv')
    model.fit(X, y)
    prediction = model.predict(test)
print('Training complete; generated', len(prediction), 'predictions.')


## 19. Check the selected features
Recover human-readable names from the fitted pipeline and require the same 324 features as the frozen full-data model.


In [ ]:
def selected_names(model, columns):
    branches = dict(model.named_steps['features'].transformer_list)
    current = dict(branches['current'].transformer_list)
    base = current['baseline']
    names = np.asarray([*columns, *base.named_steps['encoding'].vectorizer_.feature_names_], dtype=object)
    names = names[base.named_steps['constant_filter'].get_support()]
    names = list(names[base.named_steps['feature_selection'].get_support()])
    for prefix, branch in [('matchup', current['matchups']), ('counter', branches['counters'])]:
        raw_names = branch.named_steps['raw'].get_feature_names_out()
        names.extend(prefix + '__' + n for n in raw_names[branch.named_steps['constant_filter'].get_support()])
    return names


names = selected_names(model, columns)
if names != spec['selected_features']:
    raise ValueError('Selected features differ from the frozen model; use the reference runtime.')
assert len(names) == 324
print('Verified:', len(names), 'frozen features.')


## 20. Validate and save the submission
Align predictions with the sample IDs and verify the exact frozen CSV checksum before writing. Save a reproduction report and preview five predictions.


In [ ]:
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
if (list(sample) != ['battle_id', 'player_won'] or not sample.battle_id.is_unique
        or set(sample.battle_id) != set(test_ids)):
    raise ValueError('Sample submission IDs or columns do not match test data.')

# Map by battle ID so CSV order follows the official sample, even if input order changes.
submission = sample.copy()
submission['player_won'] = submission.battle_id.map(pd.Series(prediction, index=test_ids)).astype(int)
if not submission.notna().all().all() or not submission.player_won.isin([0, 1]).all():
    raise ValueError('Invalid submission predictions.')
payload = submission.to_csv(index=False, lineterminator='\n').encode()
actual = hashlib.sha256(payload).hexdigest()
if actual != spec['submission_sha256']:
    raise ValueError('Predictions differ from the frozen submission; use the reference runtime.')

# Write only after all reproduction checks pass.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
submission_path = OUTPUT_DIR / 'submission.csv'
submission_path.write_bytes(payload)
reproduction_report = {
    'model': spec['model'], 'historical_development_cv_accuracy': spec['cv_accuracy'],
    'train_rows': len(y), 'submission_rows': len(submission), 'features': len(names),
    'submission_sha256': actual, 'exact_frozen_submission_reproduced': True,
    'implementation': 'Visible notebook code',
    'versions': {name: version(name) for name in spec['versions']},
}
(OUTPUT_DIR / 'reproduction_report.json').write_text(json.dumps(reproduction_report, indent=2) + '\n')
print('Ready:', submission_path)
print('Exact frozen submission reproduced:', actual)
print(submission.head().to_string(index=False))
